# Initialization Ablation: Random vs Graph-Based

**Goal:** Show that graph structure improves embeddings

**Comparison:**
- Random-Init: Standard random initialization (torch.randn * 0.1)
- Graph-Init: Degree-based initialization (entities with more connections scaled differently)

**Hypothesis:** Graph structure in initialization improves OOD detection

In [1]:
# Setup
import os, sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

Cloning into '/content/kg-bayesian-prior'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (283/283), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 283 (delta 174), reused 197 (delta 88), pack-reused 0 (from 0)
Receiving objects: 100% (283/283), 228.78 KiB | 3.41 MiB/s, done.
Resolving deltas: 100% (174/174), done.


In [2]:
import gc, json, warnings, time
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Fast config for CPU (~30 min total)
CONFIG = {
    'embedding_dim': 50,
    'epochs': 10,
    'num_inducing': 200,
    'mrr_sample': 500,
    'kl_weight': 0.01,       # KL regularization weight
    'min_edges': 2000,       # Higher = fewer relations = faster eigendecomp
    'num_eigenvectors': 5,   # Fewer = faster eigendecomp
}
print(f"Config: {CONFIG}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

results = {}

Device: cpu
Config: {'embedding_dim': 50, 'epochs': 10, 'num_inducing': 200, 'mrr_sample': 500, 'kl_weight': 0.01, 'min_edges': 2000, 'num_eigenvectors': 5}
FB15k-237 not found. Downloading...


train.txt: 21.0MB [00:00, 88.3MB/s]


valid.txt: 1.29MB [00:00, 4.36MB/s]


test.txt: 1.51MB [00:00, 7.38MB/s]


FB15k-237 download complete!
Data: 272,115 train, 20,466 test


In [3]:
def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def evaluate(model, name):
    """Fast evaluation with small samples"""
    print(f"\nEvaluating {name}...")
    model.eval()

    # MRR (small sample)
    sample_idx = np.random.choice(len(test_data), min(CONFIG['mrr_sample'], len(test_data)), replace=False)
    sample = test_data.triples[sample_idx]

    ranks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sample), 50), desc="MRR", leave=False):
            batch = sample[i:i+50]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            scores = model.score_tails(h, r)
            target = scores[torch.arange(len(t), device=device), t]
            ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

    ranks = torch.tensor(ranks, dtype=torch.float)
    mrr = (1/ranks).mean().item()
    h1 = (ranks <= 1).float().mean().item()
    h10 = (ranks <= 10).float().mean().item()

    # ECE (small sample)
    ece_sample = 500
    ece_idx = np.random.choice(len(test_data), ece_sample, replace=False)
    pos = test_data.triples[ece_idx]
    neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
    all_t = np.vstack([pos, neg])
    labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

    with torch.no_grad():
        h, r, t = [torch.tensor(all_t[:,j], device=device) for j in range(3)]
        scores = model.score_triple(h, r, t)
        conf = torch.sigmoid(scores).cpu().numpy()
    ece, _ = expected_calibration_error(conf, labels)

    # AUROC (small sample)
    ood_sample = 500
    id_idx = np.random.choice(len(test_data), ood_sample, replace=False)
    id_triples = test_data.triples[id_idx]
    ood_triples = create_ood_dataset(train_data, test_data, "random", ood_sample)

    def get_unc(triples):
        with torch.no_grad():
            h, r, t = [torch.tensor(triples[:,j], device=device) for j in range(3)]
            pred = model.predict_with_uncertainty(h, r, t)
            return pred['total'].cpu().numpy()

    auroc = compute_auroc(get_unc(id_triples), get_unc(ood_triples))

    results[name] = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "auroc": auroc}
    print(f"{name}: MRR={mrr:.4f}, H@10={h10:.4f}, ECE={ece:.4f}, AUROC={auroc:.4f}")
    return results[name]

---
## GP-KGE (RBF Kernel) - Identity Prior
---

In [ ]:
print("="*50 + "\nGP-KGE (Random-Init)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_rbf = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="rbf",
    num_inducing=CONFIG['num_inducing']
).to(device)

print("Using random initialization")

opt = torch.optim.Adam(model_rbf.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (Random-Init)")):
    model_rbf.train()
    loss_sum, n = 0, 0
    
    for batch_idx, st in enumerate(range(0, len(train_data), 1024)):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()

        ps = model_rbf.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_rbf.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_rbf.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_rbf, "GP-KGE (Random-Init)")
del model_rbf
clear_mem()

---
## GP-KGE (Relation-Aware Kernel) - Graph Prior
---

In [ ]:
print("="*50 + "\nGP-KGE (Graph-Init)\n" + "="*50)
set_seed(42)
clear_mem()

start = time.time()
model_ra = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=CONFIG['embedding_dim'],
    kernel_type="relation_aware",
    num_inducing=CONFIG['num_inducing']
).to(device)

# Simple graph-based init: use node degree as initialization signal
print("Initializing from graph structure (degree-based)...")
try:
    # Build simple degree features from graph
    from scipy import sparse
    total_adj = sparse.csr_matrix((train_data.num_entities, train_data.num_entities))
    for rel_id, adj in train_data.relation_adjacencies.items():
        total_adj = total_adj + adj
    
    # Degree of each node
    degrees = np.array(total_adj.sum(axis=1)).flatten()
    degrees = degrees / (degrees.max() + 1e-8)  # Normalize
    
    # Use degree as initialization bias (entities with more connections get different init)
    with torch.no_grad():
        degree_tensor = torch.tensor(degrees, dtype=torch.float32).unsqueeze(1)
        # Modulate random init by degree
        model_ra.entity_mean.data = model_ra.entity_mean.data * (0.5 + degree_tensor)
    
    print(f"Initialized {train_data.num_entities} entities with degree-based scaling")
except Exception as e:
    print(f"Warning: {e}")
    print("Using random init")

In [ ]:
# Training (same as RBF, no KL - just BCE)
opt = torch.optim.Adam(model_ra.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(CONFIG['epochs']), desc="GP-KGE (Graph-Init)")):
    model_ra.train()
    loss_sum, n = 0, 0
    
    for batch_idx, st in enumerate(range(0, len(train_data), 1024)):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()

        ps = model_ra.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model_ra.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ra.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print(f"Training time: {time.time()-start:.1f}s")
evaluate(model_ra, "GP-KGE (Graph-Init)")
del model_ra
clear_mem()

---
## Results
---

In [ ]:
print("\n" + "="*70)
print("KERNEL ABLATION RESULTS")
print("="*70)
print(f"{'Model':<25} {'MRR':>8} {'H@1':>8} {'H@10':>8} {'ECE':>8} {'AUROC':>8}")
print("-"*70)

for name, r in results.items():
    print(f"{name:<25} {r['mrr']:>8.4f} {r['hits@1']:>8.4f} {r['hits@10']:>8.4f} {r['ece']:>8.4f} {r['auroc']:>8.4f}")

if len(results) == 2:
    rand = results.get("GP-KGE (Random-Init)", {})
    graph = results.get("GP-KGE (Graph-Init)", {})
    if rand and graph:
        print("\n" + "="*70)
        print("IMPROVEMENT (Graph-Init vs Random-Init)")
        print("="*70)
        print(f"  MRR:   {rand['mrr']:.4f} -> {graph['mrr']:.4f} ({(graph['mrr']-rand['mrr'])/rand['mrr']*100:+.1f}%)")
        print(f"  H@10:  {rand['hits@10']:.4f} -> {graph['hits@10']:.4f} ({(graph['hits@10']-rand['hits@10'])/rand['hits@10']*100:+.1f}%)")
        print(f"  AUROC: {rand['auroc']:.4f} -> {graph['auroc']:.4f} ({(graph['auroc']-rand['auroc'])/rand['auroc']*100:+.1f}%)")
        print(f"  ECE:   {rand['ece']:.4f} -> {graph['ece']:.4f} ({(rand['ece']-graph['ece'])/rand['ece']*100:+.1f}% better)")

In [ ]:
# Save
with open('kernel_ablation_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to kernel_ablation_results.json")

try:
    from google.colab import files
    files.download('kernel_ablation_results.json')
except:
    pass